In [0]:
table_name = "fact_encounters"
silver_table_fqn = f"hdfc_ergo.hdfc_silver.{table_name}"

df = spark.read.table("hdfc_ergo.hdfc_bronze.fact_encounters")

In [0]:
from pyspark.sql.functions import col, upper, trim, when, to_date, try_to_timestamp, row_number, abs as spark_abs
from pyspark.sql.window import Window

# 1. Enrich: Map Source SKs -> Business Keys
df_map_member = spark.read.table("hdfc_ergo.hdfc_bronze.dim_members").select(col("member_sk").cast("int").alias("src_member_sk"), col("member_id"))
df_map_provider = spark.read.table("hdfc_ergo.hdfc_bronze.dim_providers").select(col("provider_sk").cast("int").alias("src_provider_sk"), col("provider_id"))
df_map_facility = spark.read.table("hdfc_ergo.hdfc_bronze.dim_facilities").select(col("facility_sk").cast("int").alias("src_facility_sk"), col("facility_id"))
df_map_service = spark.read.table("hdfc_ergo.hdfc_bronze.dim_service_types").select(col("service_type_sk").cast("int").alias("src_service_type_sk"), col("service_type_id"))
df_map_ref_provider = spark.read.table("hdfc_ergo.hdfc_bronze.dim_providers").select(col("provider_sk").cast("int").alias("src_referring_provider_sk"), col("provider_id").alias("referring_provider_id"))

df = df.join(df_map_member, df["member_sk"] == df_map_member["src_member_sk"], "left").drop("src_member_sk", "member_sk")
df = df.join(df_map_provider, df["provider_sk"] == df_map_provider["src_provider_sk"], "left").drop("src_provider_sk", "provider_sk")
df = df.join(df_map_facility, df["facility_sk"] == df_map_facility["src_facility_sk"], "left").drop("src_facility_sk", "facility_sk")
df = df.join(df_map_service, df["service_type_sk"] == df_map_service["src_service_type_sk"], "left").drop("src_service_type_sk", "service_type_sk")
df = df.join(df_map_ref_provider, df["referring_provider_sk"] == df_map_ref_provider["src_referring_provider_sk"], "left").drop("src_referring_provider_sk", "referring_provider_sk")

# 2. Standardize Text
for c in ["encounter_id", "member_id", "provider_id", "facility_id", "service_type_id", "referring_provider_id", "encounter_type", "cancellation_reason", "provider_notes"]:
    df = df.withColumn(c, upper(trim(col(c))))

# 3. Safe Cast Dates & Timestamps
df = df.withColumn("encounter_date", to_date(trim(col("encounter_date"))))

# FIX: Use try_to_timestamp to safely turn 'N/A' or malformed text into NULL instead of crashing
for ts_col in ["encounter_start_time", "encounter_end_time", "check_in_time", "check_out_time", "record_created_date", "record_modified_date"]:
    df = df.withColumn(ts_col, try_to_timestamp(trim(col(ts_col))))

# 4. Safe Cast Integers
for int_col in ["duration_minutes", "wait_time_minutes", "diagnosis_count", "procedure_count", "medication_count"]:
    df = df.withColumn(int_col, trim(col(int_col)).cast("int"))

# 5. Safe Cast Decimals
is_valid = trim(col("patient_satisfaction")).rlike("^[+-]?([0-9]+\\.?[0-9]*|[0-9]*\\.?[0-9]+)$")
df = df.withColumn("patient_satisfaction", when(is_valid, trim(col("patient_satisfaction")).cast("decimal(5,2)")).otherwise(None))

# 6. Boolean Mapping
for bool_col in ["vital_signs_recorded", "no_show", "follow_up_required"]:
    df = df.withColumn(bool_col, when(upper(trim(col(bool_col))).isin("YES", "1", "TRUE"), True).when(upper(trim(col(bool_col))).isin("NO", "0", "FALSE"), False).otherwise(None).cast("boolean"))

# 7. Deduplication & Drop Audit
dedup_window = Window.partitionBy("encounter_id").orderBy(col("_ingested_at").desc())
df = df.withColumn("_dq_is_deduped", row_number().over(dedup_window)).filter(col("_dq_is_deduped") == 1).drop("_dq_is_deduped", "_ingested_at", "_source_file", "encounter_sk")

display(df.limit(5))

In [0]:
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(silver_table_fqn)
print(f"✅ Successfully wrote {table_name} to Silver layer.")